In [78]:
import pandas as pd
import numpy as np
df = pd.read_csv('Employee_Dataset.csv')
df.head()

,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active
0,EMP1000,IT,Senior Analyst,60,85000,invalid,01/04/2021,5,3,NaN
1,emp_1,IT,Analyst,NaN,55000,10/06/2020,invalid,5,3,True
2,EMP1002,Sales,NaN,28,85000,NaN,2022-03-01,3,4,NaN
3,EMP1003,it,Analyst,45,35000,invalid,01/04/2021,1,3,NaN
4,NaN,IT,mgr,150,85000,NaN,invalid,12,2,True


In [79]:

# =============================
# DATA CLEANING & VALIDATION
# =============================

# 1. Convert joining_date
df['joining_date_clean'] = pd.to_datetime(df['joining_date'], errors='coerce')
failed_joining_date = df['joining_date_clean'].isna().sum()



C:\Users\Sathvik\AppData\Local\Temp\ipykernel_177568\455481182.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['joining_date_clean'] = pd.to_datetime(df['joining_date'], errors='coerce')


In [80]:
# 2. Clean employee_id
df['employee_id_clean'] = df['employee_id'].astype(str).str.strip().str.upper()
duplicate_employees = df['employee_id_clean'].duplicated().sum()

In [81]:

# 3. Standardize department & salary
df['department_clean'] = df['department'].str.strip().str.title()
df['salary_clean'] = pd.to_numeric(df['salary'], errors='coerce')
avg_salary_dept = df.groupby('department_clean')['salary_clean'].mean()

In [ ]:
df['age_clean'] = pd.to_numeric(df['age'], errors='coerce')
invalid_age_valid_salary = df[(df['salary_clean'].notna()) & (df['age_clean'].isna())]


In [83]:
Q1 = df['salary_clean'].quantile(0.25)
Q3 = df['salary_clean'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
salary_outliers = df[(df['salary_clean'] < lower) | (df['salary_clean'] > upper)]


In [84]:
# 6. Performance rating median
df['performance_rating_clean'] = pd.to_numeric(df['performance_rating'], errors='coerce')
df['designation_clean'] = df['designation'].str.strip().str.title()
median_rating = df.groupby('designation_clean')['performance_rating_clean'].median()

In [85]:
# 7. Invalid promotion dates
df['last_promotion_date_clean'] = pd.to_datetime(df['last_promotion_date'], errors='coerce')
invalid_promotion = df[df['last_promotion_date_clean'] < df['joining_date_clean']]

In [86]:



# 8. Experience mismatch
df['experience_clean'] = pd.to_numeric(df['experience_years'], errors='coerce')
experience_age_mismatch = df[df['experience_clean'] > df['age_clean']]

In [87]:
# 9. Active employees count
df['is_active_clean'] = df['is_active'].astype(str).str.lower().map({'true':True,'false':False})
active_count = df[df['is_active_clean'] == True].groupby('designation_clean').size()

In [88]:

# 10. Inactive with recent promotions
recent_cutoff = pd.Timestamp.today() - pd.DateOffset(years=2)
inactive_recent_promo = df[(df['is_active_clean'] == False) & 
                           (df['last_promotion_date_clean'] >= recent_cutoff)]

In [89]:

# 11. Tenure calculation
df['tenure_years'] = (pd.Timestamp.today() - df['joining_date_clean']).dt.days / 365
p90 = df['tenure_years'].quantile(0.9)
high_tenure = df[df['tenure_years'] > p90]

In [90]:
# 12. Dept >25% missing salary
salary_missing_pct = df.groupby('department_clean')['salary_clean'].apply(lambda x: x.isna().mean())
dept_high_missing = salary_missing_pct[salary_missing_pct > 0.25]

In [91]:
# 13. High performance but low salary
median_salary = df['salary_clean'].median()
df['high_perf_low_salary_flag'] = np.where(
    (df['performance_rating_clean'] >= 4) & 
    (df['salary_clean'] < median_salary), 1, 0)
print(df['high_perf_low_salary_flag'].value_counts())

high_perf_low_salary_flag
0    942
1     58
Name: count, dtype: int64


In [92]:

# 14. No promotion but >5 years exp
no_promo_high_exp = df[(df['last_promotion_date_clean'].isna()) & 
                       (df['experience_clean'] > 5)]

In [93]:
# 15. Multi-constraint violation
df['constraint1'] = df['joining_date_clean'].isna()
df['constraint2'] = df['salary_clean'].isna()
df['constraint3'] = df['age_clean'].isna()
df['violation_count'] = df[['constraint1','constraint2','constraint3']].sum(axis=1)
df['multi_violation_flag'] = df['violation_count'] >= 2
df

,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active,...,last_promotion_date_clean,experience_clean,is_active_clean,tenure_years,high_perf_low_salary_flag,constraint1,constraint2,constraint3,violation_count,multi_violation_flag
0,EMP1000,IT,Senior Analyst,60,85000,invalid,01/04/2021,5,3,NaN,...,2021-01-04,5.0,NaN,NaN,0,True,False,False,1,False
1,emp_1,IT,Analyst,NaN,55000,10/06/2020,invalid,5,3,True,...,NaT,5.0,True,5.353425,0,False,False,True,1,False
2,EMP1002,Sales,NaN,28,85000,NaN,2022-03-01,3,4,NaN,...,NaT,3.0,NaN,NaN,0,True,False,False,1,False
3,EMP1003,it,Analyst,45,35000,invalid,01/04/2021,1,3,NaN,...,2021-01-04,1.0,NaN,NaN,0,True,False,False,1,False
4,NaN,IT,mgr,150,85000,NaN,invalid,12,2,True,...,NaT,12.0,True,NaN,0,True,False,False,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,EMP1995,Finance,NaN,60,55000,2019-05-10,2022-03-01,1,4,yes,...,NaT,1.0,NaN,6.764384,1,False,False,False,0,False
996,EMP1996,IT,Analyst,35,250000,invalid,invalid,-2,5,NaN,...,NaT,-2.0,NaN,NaN,0,True,False,False,1,False
997,EMP1997,Sales,Senior Analyst,150,250000,2021/07/15,invalid,3,excellent,yes,...,NaT,3.0,NaN,4.580822,0,False,False,False,0,False
998,emp_998,Finance,NaN,60,85000,2019-05-10,NaN,12,NaN,True,...,NaT,12.0,True,6.764384,0,False,False,False,0,False


In [94]:
df['last_promotion_date_std'] = pd.to_datetime(df['last_promotion_date'], errors='coerce', dayfirst=True)

df['joining_date_std'] = pd.to_datetime(df['joining_date'],errors='coerce',dayfirst=True)

promotion_gap = df[(df['last_promotion_date_std'] - df['joining_date_std']).dt.days/365.25 < 1]
promotion_gap

C:\Users\Sathvik\AppData\Local\Temp\ipykernel_177568\3329214143.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['joining_date_std'] = pd.to_datetime(df['joining_date'],errors='coerce',dayfirst=True)


,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active,...,is_active_clean,tenure_years,high_perf_low_salary_flag,constraint1,constraint2,constraint3,violation_count,multi_violation_flag,last_promotion_date_std,joining_date_std
9,EMP1009,it,NaN,22,120000,10/06/2020,01/04/2021,NaN,4,NaN,...,NaN,5.353425,0,False,False,False,0,False,2021-04-01,2020-06-10
24,EMP1024,Finance,mgr,35,85000,2021/07/15,01/04/2021,8,3,True,...,True,4.580822,0,False,False,False,0,False,2021-04-01,2021-07-15
40,emp_40,NaN,NaN,60,85000,2021/07/15,01/04/2021,12,5,True,...,True,4.580822,0,False,False,False,0,False,2021-04-01,2021-07-15
44,emp_44,sales,mgr,45,₹75000,2021/07/15,01/04/2021,8,2,True,...,True,4.580822,0,False,True,False,1,False,2021-04-01,2021-07-15
47,NaN,Finance,Analyst,unknown,250000,2021/07/15,01/04/2021,3,excellent,True,...,True,4.580822,0,False,False,True,1,False,2021-04-01,2021-07-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
943,EMP1943,Finance,mgr,45,35000,10/06/2020,01/04/2021,-2,NaN,NaN,...,NaN,5.353425,0,False,False,False,0,False,2021-04-01,2020-06-10
945,NaN,HR,Manager,-5,35000,10/06/2020,01/04/2021,3,NaN,True,...,True,5.353425,0,False,False,False,0,False,2021-04-01,2020-06-10
966,NaN,it,Analyst,45,NaN,10/06/2020,01/04/2021,1,NaN,no,...,NaN,5.353425,0,False,True,False,1,False,2021-04-01,2020-06-10
974,emp_974,NaN,mgr,unknown,85000,2021/07/15,01/04/2021,3,4,False,...,False,4.580822,0,False,False,True,1,False,2021-04-01,2021-07-15


In [95]:
df['sal_std'] = (df['salary'].astype(str).str.replace('₹', '', regex=False).str.replace(',', '', regex=False))
df['sal_std'] = pd.to_numeric(df['sal_std'], errors='coerce')
df['dept_std'] = df['department'].astype(str).str.strip().str.lower()
df['performance_rating_num'] = pd.to_numeric(df['performance_rating'], errors='coerce')
df['dept_avg_salary'] = (df.groupby('dept_std')['sal_std'].transform('mean'))
df['dept_median_perf'] = (df.groupby('dept_std')['performance_rating_num'].transform('median'))
result = df[(df['sal_std'] > df['dept_avg_salary']) & (df['performance_rating_num'] < df['dept_median_perf'])]
result

,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active,...,constraint3,violation_count,multi_violation_flag,last_promotion_date_std,joining_date_std,sal_std,dept_std,performance_rating_num,dept_avg_salary,dept_median_perf
45,NaN,Finance,NaN,NaN,120000,NaN,01/04/2021,1,1,yes,...,True,2,True,2021-04-01,NaT,120000.0,finance,1.0,107397.959184,3.0
89,emp_89,NaN,Senior Analyst,22,120000,10/06/2020,NaN,1,1,no,...,False,0,False,NaT,2020-06-10,120000.0,nan,1.0,96039.603960,3.0
115,emp_115,sales,ANALYST,28,250000,2021/07/15,invalid,-2,2,no,...,False,0,False,NaT,2021-07-15,250000.0,sales,2.0,110842.696629,3.0
127,NaN,Finance,NaN,150,120000,invalid,2022-03-01,3,1,True,...,False,1,False,NaT,NaT,120000.0,finance,1.0,107397.959184,3.0
157,NaN,Finance,ANALYST,22,250000,2019-05-10,2022-03-01,3,1,False,...,False,0,False,NaT,2019-05-10,250000.0,finance,1.0,107397.959184,3.0
169,emp_169,it,mgr,60,120000,2021/07/15,01/04/2021,NaN,2,NaN,...,False,0,False,2021-04-01,2021-07-15,120000.0,it,2.0,103373.205742,3.0
173,emp_173,Finance,Manager,45,120000,10/06/2020,invalid,8,1,False,...,False,0,False,NaT,2020-06-10,120000.0,finance,1.0,107397.959184,3.0
216,emp_216,NaN,Senior Analyst,60,120000,invalid,invalid,8,2,NaN,...,False,1,False,NaT,NaT,120000.0,nan,2.0,96039.603960,3.0
217,NaN,it,NaN,28,250000,2021/07/15,2022-03-01,NaN,2,yes,...,False,0,False,NaT,2021-07-15,250000.0,it,2.0,103373.205742,3.0
223,NaN,HR,Manager,NaN,120000,2019-05-10,NaN,-2,1,True,...,True,1,False,NaT,2019-05-10,120000.0,hr,1.0,110117.647059,3.0


In [96]:
#3) Create a column that categorizes employees into age bands (Young, Mid, Senior) and count employees per band per department.
df['age_num'] = pd.to_numeric(df['age'], errors='coerce')

df['age_band'] = pd.cut(
    df['age_num'],
    bins=[0, 29, 45, np.inf],
    labels=['Young', 'Mid', 'Senior']
)

df['dept_std'] = df['department'].astype(str).str.strip().str.lower()

age_band_count = (
    df
    .groupby(['dept_std', 'age_band'], observed=True)
    .size()
    .reset_index(name='employee_count')
)

age_band_count

,dept_std,age_band,employee_count
0,finance,Young,33
1,finance,Mid,31
2,finance,Senior,24
3,hr,Young,54
4,hr,Mid,52
5,hr,Senior,36
6,it,Young,53
7,it,Mid,78
8,it,Senior,58
9,nan,Young,26


In [97]:
#4) Find departments where the average experience is higher than the company-wide average experience.

df['experience_num'] = pd.to_numeric(df['experience_years'], errors='coerce')

# Standardize department names
df['dept_std'] = df['department'].astype(str).str.strip().str.lower()

# Company-wide average experience
company_avg_experience = df['experience_num'].mean()

# Department-wise average experience
dept_avg_experience = (df.groupby('dept_std', observed=True)['experience_num'].mean().reset_index(name='dept_avg_experience'))

# Filter departments above company average
result = dept_avg_experience[dept_avg_experience['dept_avg_experience'] > company_avg_experience]

# Display result
result

,dept_std,dept_avg_experience
0,finance,4.826923
4,sales,4.709677


In [98]:
#6 Identify designations where more than 20% of employees have missing or invalid experience values. 
# Clean experience
df['experience_num'] = pd.to_numeric(df['experience_years'], errors='coerce')

# Standardize designation
df['designation_std'] = df['designation'].astype(str).str.strip().str.lower()

# Calculate invalid experience percentage per designation
designation_exp_issue = (
    df
    .assign(invalid_exp=lambda x: x['experience_num'].isna())
    .groupby('designation_std', observed=True)
    .agg(
        total_employees=('experience_num', 'size'),
        invalid_experience=('invalid_exp', 'sum')
    )
    .assign(
        invalid_pct=lambda x: x['invalid_experience'] / x['total_employees']
    )
    .reset_index()
)

# Filter > 20%
designation_exp_issue[designation_exp_issue['invalid_pct'] > 0.20]

,designation_std,total_employees,invalid_experience,invalid_pct
2,mgr,178,47,0.264045
4,senior analyst,188,38,0.202128


In [99]:
#7 Calculate the ratio of salary to experience years and detect extreme values using the 95th percentile. 
# Clean salary
df['salary_num'] = (
    df['salary']
    .astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace(',', '', regex=False)
)
df['salary_num'] = pd.to_numeric(df['salary_num'], errors='coerce')

# Salary to experience ratio
df['sal_exp_ratio'] = df['salary_num'] / df['experience_num']

# 95th percentile threshold
ratio_95 = df['sal_exp_ratio'].quantile(0.95)

# Extreme values
extreme_ratio_employees = df[df['sal_exp_ratio'] > ratio_95]

extreme_ratio_employees

,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active,...,dept_std,performance_rating_num,dept_avg_salary,dept_median_perf,age_num,age_band,experience_num,designation_std,salary_num,sal_exp_ratio
124,NaN,sales,Analyst,unknown,250000,2021/07/15,01/04/2021,1,5,no,...,sales,5.0,110842.696629,3.0,NaN,NaN,1.0,analyst,250000.0,250000.0
133,NaN,it,NaN,-5,250000,2021/07/15,2022-03-01,1,5,True,...,it,5.0,103373.205742,3.0,-5.0,NaN,1.0,nan,250000.0,250000.0
176,NaN,it,Analyst,22,250000,invalid,NaN,1,NaN,False,...,it,NaN,103373.205742,3.0,22.0,Young,1.0,analyst,250000.0,250000.0
224,NaN,Finance,Manager,60,250000,2021/07/15,invalid,1,3,NaN,...,finance,3.0,107397.959184,3.0,60.0,Senior,1.0,manager,250000.0,250000.0
262,NaN,IT,Manager,-5,250000,NaN,invalid,1,5,NaN,...,it,5.0,103373.205742,3.0,-5.0,NaN,1.0,manager,250000.0,250000.0
399,NaN,Finance,ANALYST,60,250000,invalid,NaN,1,3,True,...,finance,3.0,107397.959184,3.0,60.0,Senior,1.0,analyst,250000.0,250000.0
401,EMP1401,Finance,Analyst,22,250000,NaN,invalid,1,1,NaN,...,finance,1.0,107397.959184,3.0,22.0,Young,1.0,analyst,250000.0,250000.0
470,EMP1470,HR,mgr,35,250000,2021/07/15,2022-03-01,1,excellent,True,...,hr,NaN,110117.647059,3.0,35.0,Mid,1.0,mgr,250000.0,250000.0
525,emp_525,sales,Manager,-5,250000,10/06/2020,2022-03-01,1,1,False,...,sales,1.0,110842.696629,3.0,-5.0,NaN,1.0,manager,250000.0,250000.0
529,emp_529,NaN,NaN,28,250000,10/06/2020,invalid,1,5,no,...,nan,5.0,96039.603960,3.0,28.0,Young,1.0,nan,250000.0,250000.0


In [100]:
#8 Find employees whose age is valid but inconsistent with experience (experience > age − 18). 
# Clean age
df['age_num'] = pd.to_numeric(df['age'], errors='coerce')

# Inconsistent rows
age_exp_inconsistent = df[
    df['age_num'].notna() &
    df['experience_num'].notna() &
    (df['experience_num'] > (df['age_num'] - 18))
]

age_exp_inconsistent

,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active,...,dept_std,performance_rating_num,dept_avg_salary,dept_median_perf,age_num,age_band,experience_num,designation_std,salary_num,sal_exp_ratio
10,emp_10,Finance,Analyst,-5,35000,NaN,NaN,8,1,no,...,finance,1.0,107397.959184,3.0,-5.0,NaN,8.0,analyst,35000.0,4375.000000
17,emp_17,sales,Senior Analyst,-5,NaN,NaN,01/04/2021,3,4,False,...,sales,4.0,110842.696629,3.0,-5.0,NaN,3.0,senior analyst,NaN,NaN
23,emp_23,Finance,NaN,-5,85000,invalid,NaN,-2,5,NaN,...,finance,5.0,107397.959184,3.0,-5.0,NaN,-2.0,nan,85000.0,-42500.000000
43,NaN,it,Manager,-5,120000,invalid,NaN,1,excellent,NaN,...,it,NaN,103373.205742,3.0,-5.0,NaN,1.0,manager,120000.0,120000.000000
49,NaN,it,mgr,-5,55000,2021/07/15,invalid,5,3,False,...,it,3.0,103373.205742,3.0,-5.0,NaN,5.0,mgr,55000.0,11000.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
968,emp_968,IT,Manager,22,120000,2021/07/15,2022-03-01,5,poor,True,...,it,NaN,103373.205742,3.0,22.0,Young,5.0,manager,120000.0,24000.000000
981,NaN,sales,mgr,-5,85000,NaN,NaN,-2,NaN,yes,...,sales,NaN,110842.696629,3.0,-5.0,NaN,-2.0,mgr,85000.0,-42500.000000
985,emp_985,sales,mgr,-5,NaN,invalid,invalid,-2,1,NaN,...,sales,1.0,110842.696629,3.0,-5.0,NaN,-2.0,mgr,NaN,NaN
990,emp_990,NaN,Senior Analyst,-5,85000,2019-05-10,01/04/2021,12,5,yes,...,nan,5.0,96039.603960,3.0,-5.0,NaN,12.0,senior analyst,85000.0,7083.333333


In [102]:
# Standardize department
df['dept_std'] = df['department'].astype(str).str.strip().str.lower()

# Attrition calculation (fixed: removed extra ))
dept_attrition = (
    df
    .groupby('dept_std', observed=True)
    .agg(
        total_employees=('employee_id', 'count'),
        inactive_employees=('is_active_bool', lambda x: (~x).sum())
    )
    .assign(attrition_rate=lambda x: x['inactive_employees'] / x['total_employees'])
    .reset_index()
)

dept_attrition

KeyError: "Column(s) ['is_active_bool'] do not exist"

In [103]:
#10 Create a final quality score per row based on number of valid fields and flag rows below a quality threshold. 

# Clean performance rating
df['performance_rating_num'] = pd.to_numeric(
    df['performance_rating'], errors='coerce'
)

# Define valid field checks
valid_fields = {
    'age': df['age_num'].notna(),
    'experience': df['experience_num'].notna(),
    'salary': df['salary_num'].notna(),
    'department': df['department'].notna(),
    'designation': df['designation'].notna(),
    'performance_rating': df['performance_rating_num'].notna()
}

# Compute quality score
df['quality_score'] = sum(valid_fields.values())

# Define threshold (example: at least 4 valid fields)
quality_threshold = 4

# Flag low-quality rows
df['low_quality_flag'] = df['quality_score'] < quality_threshold

df[['employee_id', 'quality_score', 'low_quality_flag']]

,employee_id,quality_score,low_quality_flag
0,EMP1000,6,False
1,emp_1,5,False
2,EMP1002,5,False
3,EMP1003,6,False
4,NaN,6,False
...,...,...,...
995,EMP1995,5,False
996,EMP1996,6,False
997,EMP1997,5,False
998,emp_998,4,False
